# IPL Match Winning Probability Prediction using Artificial Neural Network (ANN)

**Level:** Advanced | **Platform:** Google Colab | **Task:** Predict the chasing team's probability of winning after each delivery in the second innings.

## Project workflow
1. Download historical IPL ball-by-ball JSON data from Cricsheet.
2. Extract match and delivery records.
3. Build second-innings match-state snapshots and labels.
4. Clean and explore the data.
5. Engineer cricket features (target, runs remaining, balls remaining, run rates, wickets).
6. Split by match ID to prevent deliveries from the same match leaking across train/test.
7. Train an ANN with TensorFlow/Keras.
8. Evaluate, save, and test the model with a custom prediction function.

**Important:** This project predicts the *chasing team's* win probability. It is not a guaranteed prediction. Historical data may not represent current team strength, player form, or every rule change.

## 1. Install and import libraries

Run this notebook from top to bottom in Google Colab. If Colab asks you to restart after installation, restart the runtime and continue.

In [3]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn tensorflow requests joblib


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\csgau\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [4]:
import os
import json
import zipfile
import requests
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Python libraries loaded")
print("TensorFlow:", tf.__version__)

ModuleNotFoundError: No module named 'requests'


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\csgau\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


## 2. Download IPL data from Cricsheet

Source: [Cricsheet downloads](https://cricsheet.org/downloads/). The notebook downloads the IPL JSON archive directly. Please check Cricsheet's terms and data notes before redistributing the dataset.

In [ ]:
DATA_URL = "https://cricsheet.org/downloads/ipl_json.zip"
ZIP_PATH = "ipl_json.zip"
DATA_DIR = "ipl_json"

if not os.path.exists(ZIP_PATH):
    response = requests.get(DATA_URL, timeout=180)
    response.raise_for_status()
    with open(ZIP_PATH, "wb") as f:
        f.write(response.content)
    print("Downloaded:", ZIP_PATH)
else:
    print("Archive already exists:", ZIP_PATH)

os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(DATA_DIR)

json_files = [
    os.path.join(DATA_DIR, name)
    for name in os.listdir(DATA_DIR)
    if name.lower().endswith(".json")
]
print("JSON match files:", len(json_files))
if not json_files:
    raise RuntimeError("No JSON files found. Check the download/archive structure.")

## 3. Inspect one raw match

Cricsheet JSON stores match metadata, innings, overs, and deliveries. The next cell prints a small sample so you can understand the source structure.

In [ ]:
with open(json_files[0], "r", encoding="utf-8") as f:
    sample_match = json.load(f)

print("Metadata keys:", sample_match.get("info", {}).keys())
print("Teams:", sample_match.get("info", {}).get("teams"))
print("Number of innings:", len(sample_match.get("innings", [])))
print("First innings keys:", sample_match.get("innings", [])[0].keys())
print("First over sample:", sample_match.get("innings", [])[0]["overs"][0])

## 4. Convert JSON files into match and delivery tables

This parser creates:
- `matches_df`: one row per match, with the winner.
- `deliveries_df`: one row per recorded delivery.

We retain the first two innings for the chase model. Matches without a recorded winner are retained in the raw delivery table but will not create labelled training examples.

In [ ]:
delivery_rows = []
match_rows = []

for filepath in json_files:
    match_id = os.path.splitext(os.path.basename(filepath))[0]

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print("Skipping unreadable file:", filepath, e)
        continue

    info = data.get("info", {})
    teams = info.get("teams", [])
    if len(teams) != 2:
        continue

    outcome = info.get("outcome", {})
    winner = outcome.get("winner")

    match_rows.append({
        "match_id": match_id,
        "season": str(info.get("season", "Unknown")),
        "venue": info.get("venue", "Unknown"),
        "team1": teams[0],
        "team2": teams[1],
        "winner": winner,
        "outcome_method": outcome.get("method")
    })

    for innings_no, innings in enumerate(data.get("innings", []), start=1):
        if innings_no > 2:
            continue

        batting_team = innings.get("team", "Unknown")
        target_info = innings.get("target", {}) or {}

        for over_obj in innings.get("overs", []):
            over_number = int(over_obj.get("over", 0))

            for ball_no, delivery in enumerate(over_obj.get("deliveries", []), start=1):
                runs = delivery.get("runs", {})
                extras = delivery.get("extras", {}) or {}
                wickets = delivery.get("wickets", []) or []

                delivery_rows.append({
                    "match_id": match_id,
                    "innings": innings_no,
                    "season": str(info.get("season", "Unknown")),
                    "venue": info.get("venue", "Unknown"),
                    "batting_team": batting_team,
                    "bowling_team": next(
                        (team for team in teams if team != batting_team), "Unknown"
                    ),
                    "over": over_number,
                    "ball_in_over": ball_no,
                    "batter": delivery.get("batter"),
                    "bowler": delivery.get("bowler"),
                    "batter_runs": int(runs.get("batter", 0)),
                    "total_runs": int(runs.get("total", 0)),
                    "extras_total": int(runs.get("extras", 0)),
                    "wide": int(extras.get("wides", 0) or 0),
                    "noballs": int(extras.get("noballs", 0) or 0),
                    "wicket_count": len(wickets),
                    "target_from_source": target_info.get("runs")
                })

matches_df = pd.DataFrame(match_rows)
deliveries_df = pd.DataFrame(delivery_rows)

print("Matches:", matches_df.shape)
print("Deliveries:", deliveries_df.shape)
display(matches_df.head())
display(deliveries_df.head())

## 5. Basic data quality checks

Check missing values, duplicate records, data types, and the number of matches with a known winner.

In [ ]:
print("Match table missing values:")
display(matches_df.isna().sum().to_frame("missing_count"))

print("Delivery table missing values:")
display(deliveries_df.isna().sum().to_frame("missing_count"))

print("Exact duplicate delivery rows:", deliveries_df.duplicated().sum())
print("Matches with a known winner:", matches_df["winner"].notna().sum())
print("Teams:", sorted(deliveries_df["batting_team"].dropna().unique())[:30])

## 6. Create second-innings match-state snapshots

Each training row represents the state **after one delivery** in the second innings. The target is `chase_win`:
- `1` = the chasing/batting team eventually won.
- `0` = the chasing/batting team did not win.

We calculate legal balls using wides and no-balls. The target is normally first-innings total + 1; when Cricsheet supplies a second-innings target, that value is used. For a clean baseline, matches recorded as D/L method are excluded because revised targets can require more detailed handling.

In [ ]:
winner_map = matches_df.set_index("match_id")["winner"].to_dict()
method_map = matches_df.set_index("match_id")["outcome_method"].to_dict()

snapshot_rows = []

for match_id, match_deliveries in deliveries_df.groupby("match_id", sort=False):
    if winner_map.get(match_id) is None:
        continue
    if str(method_map.get(match_id, "")).upper() in {"D/L", "DL"}:
        continue

    first = match_deliveries[match_deliveries["innings"] == 1]
    second = match_deliveries[match_deliveries["innings"] == 2]

    if first.empty or second.empty:
        continue

    first_innings_total = int(first["total_runs"].sum())
    batting_team = second["batting_team"].iloc[0]
    bowling_team = second["bowling_team"].iloc[0]
    winner = winner_map[match_id]

    source_targets = second["target_from_source"].dropna()
    target = int(source_targets.iloc[0]) if not source_targets.empty else first_innings_total + 1

    score = 0
    wickets = 0
    legal_balls = 0

    for _, ball in second.iterrows():
        score += int(ball["total_runs"])
        wickets += int(ball["wicket_count"])

        # A legal ball is one that is neither a wide nor a no-ball.
        if int(ball["wide"]) == 0 and int(ball["noballs"]) == 0:
            legal_balls += 1

        balls_remaining = max(0, 120 - legal_balls)
        runs_remaining = max(0, target - score)
        overs_completed = legal_balls / 6
        current_run_rate = score / overs_completed if overs_completed > 0 else 0.0
        required_run_rate = (
            runs_remaining / (balls_remaining / 6)
            if balls_remaining > 0 else 0.0
        )

        snapshot_rows.append({
            "match_id": match_id,
            "season": str(second["season"].iloc[0]),
            "venue": str(second["venue"].iloc[0]),
            "batting_team": str(batting_team),
            "bowling_team": str(bowling_team),
            "target": target,
            "current_score": score,
            "wickets_lost": min(wickets, 10),
            "legal_balls_bowled": legal_balls,
            "balls_remaining": balls_remaining,
            "runs_remaining": runs_remaining,
            "current_run_rate": current_run_rate,
            "required_run_rate": required_run_rate,
            "chase_win": int(winner == batting_team)
        })

model_df = pd.DataFrame(snapshot_rows)

if model_df.empty:
    raise RuntimeError("No labelled chase snapshots were created. Inspect the raw data/parser.")

print("Model dataset shape:", model_df.shape)
display(model_df.head())
print("\nTarget distribution:")
display(model_df["chase_win"].value_counts(normalize=True).rename("proportion"))

## 7. Clean and validate the modelling table

We remove invalid states, cap impossible values, and remove duplicate snapshots. We do not use batter/bowler names in this first model because they create very high-cardinality features and may make historical player identity dominate the model.

In [ ]:
model_df = model_df.replace([np.inf, -np.inf], np.nan)

numeric_cols = [
    "target", "current_score", "wickets_lost", "legal_balls_bowled",
    "balls_remaining", "runs_remaining", "current_run_rate",
    "required_run_rate"
]
categorical_cols = ["season", "venue", "batting_team", "bowling_team"]

model_df = model_df.dropna(subset=numeric_cols + categorical_cols + ["chase_win"])
model_df = model_df.drop_duplicates(subset=["match_id", "legal_balls_bowled", "current_score"])

model_df = model_df[
    (model_df["target"] > 0) &
    (model_df["current_score"] >= 0) &
    (model_df["wickets_lost"].between(0, 10)) &
    (model_df["legal_balls_bowled"].between(0, 120))
].copy()

model_df["chase_win"] = model_df["chase_win"].astype(int)

print("Clean model dataset:", model_df.shape)
print("Missing values remaining:", int(model_df.isna().sum().sum()))

## 8. Exploratory Data Analysis (EDA)

### 8.1 Target balance

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=model_df, x="chase_win")
plt.title("Chasing team match outcome distribution")
plt.xlabel("Chase win (1 = win, 0 = loss)")
plt.ylabel("Number of delivery snapshots")
plt.show()

### 8.2 Runs remaining versus winning outcome

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=model_df, x="chase_win", y="runs_remaining")
plt.title("Runs remaining by eventual chase outcome")
plt.xlabel("Chase win (1 = win, 0 = loss)")
plt.ylabel("Runs remaining after delivery")
plt.show()

### 8.3 Wickets lost versus winning outcome

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=model_df, x="wickets_lost", y="chase_win", errorbar=None)
plt.title("Historical chase win rate by wickets lost")
plt.xlabel("Wickets lost")
plt.ylabel("Snapshot win rate")
plt.show()

### 8.4 Correlation between numeric features

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(model_df[numeric_cols + ["chase_win"]].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.show()

## 9. Define features and target

The model uses only match-state information known at the prediction moment. `match_id` is used only for grouped splitting and is never fed into the ANN.

In [ ]:
FEATURES = [
    "season", "venue", "batting_team", "bowling_team",
    "target", "current_score", "wickets_lost",
    "legal_balls_bowled", "balls_remaining", "runs_remaining",
    "current_run_rate", "required_run_rate"
]
TARGET = "chase_win"

X = model_df[FEATURES].copy()
y = model_df[TARGET].copy()
groups = model_df["match_id"].copy()

print("Feature columns:", FEATURES)
print("X shape:", X.shape, "| y shape:", y.shape)

## 10. Split by match ID to avoid data leakage

A normal random row split is unsafe here: one match creates many delivery snapshots. We use `GroupShuffleSplit` so all snapshots from a match stay in only one partition. We first reserve a test set, then reserve a validation set from the remaining training matches.

In [ ]:
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=groups))

X_train_val = X.iloc[train_val_idx].copy()
y_train_val = y.iloc[train_val_idx].copy()
groups_train_val = groups.iloc[train_val_idx].copy()

X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_test = groups.iloc[test_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, val_idx = next(
    gss_val.split(X_train_val, y_train_val, groups=groups_train_val)
)

X_train = X_train_val.iloc[train_idx].copy()
y_train = y_train_val.iloc[train_idx].copy()
groups_train = groups_train_val.iloc[train_idx].copy()

X_val = X_train_val.iloc[val_idx].copy()
y_val = y_train_val.iloc[val_idx].copy()
groups_val = groups_train_val.iloc[val_idx].copy()

print("Rows | train:", len(X_train), "validation:", len(X_val), "test:", len(X_test))
print("Unique matches | train:", groups_train.nunique(),
      "validation:", groups_val.nunique(), "test:", groups_test.nunique())

## 11. Preprocess numeric and categorical features

- Numeric columns: standardization using `StandardScaler`.
- Categorical columns: one-hot encoding.
- The preprocessing pipeline is fitted on training data only, then applied to validation and test data.

In [ ]:
numeric_features = [
    "target", "current_score", "wickets_lost",
    "legal_balls_bowled", "balls_remaining", "runs_remaining",
    "current_run_rate", "required_run_rate"
]
categorical_features = ["season", "venue", "batting_team", "bowling_team"]

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    # Compatibility with older scikit-learn versions
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", encoder, categorical_features)
    ],
    remainder="drop"
)

X_train_processed = preprocessor.fit_transform(X_train).astype("float32")
X_val_processed = preprocessor.transform(X_val).astype("float32")
X_test_processed = preprocessor.transform(X_test).astype("float32")

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

## 12. Build the Artificial Neural Network (ANN)

Architecture:
- Input layer: number of processed features.
- Hidden layer 1: 128 neurons, ReLU.
- Hidden layer 2: 64 neurons, ReLU.
- Hidden layer 3: 32 neurons, ReLU.
- Dropout: regularization to reduce overfitting.
- Output: one sigmoid neuron giving the estimated probability that the chasing team wins.

In [ ]:
model = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.30),
    Dense(64, activation="relu"),
    Dropout(0.20),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()

## 13. Train the ANN

Early stopping monitors validation loss and restores the best weights. Learning-rate reduction can help when validation loss stops improving.

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

history = model.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=40,
    batch_size=256,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## 14. Visualize training performance

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df["loss"], label="Training loss")
plt.plot(history_df["val_loss"], label="Validation loss")
plt.title("ANN loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df["accuracy"], label="Training accuracy")
plt.plot(history_df["val_accuracy"], label="Validation accuracy")
plt.title("ANN accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

## 15. Evaluate on the untouched test matches

Accuracy alone is not enough. We report precision, recall, F1-score, ROC-AUC, and the confusion matrix. The 0.50 threshold is a baseline; you can tune it later using validation data, not the test set.

In [ ]:
test_probabilities = model.predict(X_test_processed, verbose=0).ravel()
test_predictions = (test_probabilities >= 0.50).astype(int)

print("Accuracy :", round(accuracy_score(y_test, test_predictions), 4))
print("Precision:", round(precision_score(y_test, test_predictions, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, test_predictions, zero_division=0), 4))
print("F1-score :", round(f1_score(y_test, test_predictions, zero_division=0), 4))

if y_test.nunique() == 2:
    print("ROC-AUC  :", round(roc_auc_score(y_test, test_probabilities), 4))

print("\nClassification report:")
print(classification_report(y_test, test_predictions, zero_division=0))

cm = confusion_matrix(y_test, test_predictions)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Test set confusion matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

if y_test.nunique() == 2:
    RocCurveDisplay.from_predictions(y_test, test_probabilities)
    plt.title("ROC curve - test set")
    plt.show()

## 16. Test a prediction with a hypothetical live match state

Change the values below to test another scenario. The categorical values should be present in training data when possible; unseen categories are accepted by the encoder but represented as all-zero columns for that category.

In [ ]:
def predict_chase_win_probability(
    batting_team,
    bowling_team,
    venue,
    season,
    target,
    current_score,
    wickets_lost,
    legal_balls_bowled
):
    balls_remaining = max(0, 120 - int(legal_balls_bowled))
    runs_remaining = max(0, int(target) - int(current_score))
    overs_completed = int(legal_balls_bowled) / 6

    current_run_rate = (
        float(current_score) / overs_completed
        if overs_completed > 0 else 0.0
    )
    required_run_rate = (
        runs_remaining / (balls_remaining / 6)
        if balls_remaining > 0 else 0.0
    )

    sample = pd.DataFrame([{
        "season": str(season),
        "venue": str(venue),
        "batting_team": str(batting_team),
        "bowling_team": str(bowling_team),
        "target": float(target),
        "current_score": float(current_score),
        "wickets_lost": float(wickets_lost),
        "legal_balls_bowled": float(legal_balls_bowled),
        "balls_remaining": float(balls_remaining),
        "runs_remaining": float(runs_remaining),
        "current_run_rate": float(current_run_rate),
        "required_run_rate": float(required_run_rate)
    }])[FEATURES]

    processed = preprocessor.transform(sample).astype("float32")
    probability = float(model.predict(processed, verbose=0)[0][0])

    return {
        "batting_team_win_probability": round(probability * 100, 2),
        "bowling_team_win_probability": round((1 - probability) * 100, 2),
        "runs_remaining": runs_remaining,
        "balls_remaining": balls_remaining,
        "current_run_rate": round(current_run_rate, 2),
        "required_run_rate": round(required_run_rate, 2)
    }

# Example only - change these values to match your selected teams/venue.
example_prediction = predict_chase_win_probability(
    batting_team="Chennai Super Kings",
    bowling_team="Mumbai Indians",
    venue="Wankhede Stadium",
    season="2024",
    target=181,
    current_score=145,
    wickets_lost=4,
    legal_balls_bowled=90
)

print(example_prediction)

## 17. Save the trained model and preprocessing pipeline

These files are required to reuse the trained model without fitting it again.

In [ ]:
model.save("ipl_ann_model.keras")
joblib.dump(preprocessor, "ipl_preprocessor.joblib")

with open("ipl_model_features.json", "w") as f:
    json.dump(FEATURES, f, indent=4)

model_df.to_csv("ipl_model_dataset.csv", index=False)

print("Saved:")
print("- ipl_ann_model.keras")
print("- ipl_preprocessor.joblib")
print("- ipl_model_features.json")
print("- ipl_model_dataset.csv")

## 18. Download project outputs from Colab

Run this cell to download the model, preprocessor, feature list, and processed dataset to your computer.

In [ ]:
from google.colab import files

for output_file in [
    "ipl_ann_model.keras",
    "ipl_preprocessor.joblib",
    "ipl_model_features.json",
    "ipl_model_dataset.csv"
]:
    files.download(output_file)

## 19. Project limitations and improvements

1. This is a historical baseline. It does not include player-level form, injuries, toss decisions, weather, or ground conditions.
2. The model is trained on second-innings delivery snapshots only.
3. The target is a match-level outcome repeated across snapshots. Later snapshots contain more information than earlier ones; evaluate performance by match and consider reporting metrics by over/phase.
4. The first-innings target logic is simplified. D/L matches are excluded, but special cases such as super overs and unusual match formats should be audited before production.
5. The dataset may contain historical team names and venue naming variations. Normalize aliases before using it in a polished app.
6. Probability calibration should be checked before presenting outputs as reliable probabilities.
7. A strong extension is to compare ANN with Logistic Regression, Random Forest, and XGBoost using the same match-level split.

## Interview explanation

"I built an IPL match winning probability prediction system using historical ball-by-ball data. I processed the JSON data, engineered live match-state features such as runs remaining, wickets lost, current run rate, and required run rate, and trained an Artificial Neural Network using TensorFlow and Keras. To avoid data leakage, I split the dataset by match ID rather than randomly splitting individual deliveries. I evaluated the model using accuracy, precision, recall, F1-score, ROC-AUC, and a confusion matrix."

## Suggested resume project title

**IPL Match Winning Probability Prediction Using Artificial Neural Networks (ANN)**

**Skills:** Python, Pandas, NumPy, SQL (optional extension), EDA, Feature Engineering, TensorFlow, Keras, Scikit-learn, Power BI (dashboard extension).